# Biohub: fixed ILP and conservative pseudo-GT analysis

This notebook **reuses the output of the previous 1500-iteration comparison notebook** mounted as a Kaggle Input. It does **not** train, run image inference, or modify that Input. It accepts either the three-seed output (`seed_42`, `seed_43`, `seed_44`) or the earlier checkpoint-comparison output (`checkpoint_comparison/iteration_1500`).

It performs four checks: candidate counts/scores per frame; the **same ILP costs** for every run (`selection_constant=7`, `track_cost=7`); a conservative filter that keeps new nodes on selected paths between **two visible-GT anchors**; and the trade-off between hidden-GT recovery and unmatched predictions. Candidate generation remains frozen. The anchor filter is a **post-filter of the selected graph**: it does not insert or force missing GT anchors into the ILP, so it can miss recoverable paths.

Also attach the Biohub competition train Input if you want hidden-GT metrics and image previews. Without it, fixed-cost and visible-anchor diagnostics still run; hidden columns remain blank. The policy grid below is fixed before the original GEFF is opened. These are **exploratory same-sequence results**; do not choose a pseudo-GT policy by maximizing hidden-GT scores in this notebook.


In [ ]:
from pathlib import Path
import os, json, subprocess, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

INPUT_ROOT = Path('/kaggle/input')
OUTPUT_DIR = Path('/kaggle/working/biohub_pseudo_analysis')
REPO = Path('/kaggle/working/linajea')
SAMPLE = '6bba_05b6850b'
BRANCH = 'biohub-io'
PINNED_COMMIT = None  # Set to a pushed SHA for a fixed code revision
OUTPUT_INPUT_DIR = None  # Or set Path('/kaggle/input/YOUR_OUTPUT_DATASET/biohub_random_hide_multiseed')
GT_DIR = Path('/kaggle/input/competitions/biohub-cell-tracking-during-development/train')
SELECTION_CONSTANT = 7.0
TRACK_COST = 7.0

def find_previous_roots(base):
    found = set()
    for current, dirs, files in os.walk(base):
        dirs[:] = [d for d in dirs if not d.endswith(('.zarr', '.geff'))]
        folder = Path(current)
        if 'candidate_nodes.csv' not in files:
            continue
        if folder.name == 'prediction' and folder.parent.name.startswith('seed_'):
            found.add(folder.parent.parent)
        elif folder.name == 'iteration_1500' and folder.parent.name == 'checkpoint_comparison':
            found.add(folder.parent.parent)
    return sorted(found)

if OUTPUT_INPUT_DIR is None:
    choices = find_previous_roots(INPUT_ROOT)
    if len(choices) != 1:
        raise RuntimeError(f'Expected one previous notebook output; found {choices}. Set OUTPUT_INPUT_DIR explicitly.')
    OUTPUT_INPUT_DIR = choices[0]
else:
    OUTPUT_INPUT_DIR = Path(OUTPUT_INPUT_DIR)
if not (GT_DIR / f'{SAMPLE}.geff').is_dir():
    print('Original GEFF is not attached; hidden-GT metrics and image previews will be skipped.')
    GT_DIR = None
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Previous output Input:', OUTPUT_INPUT_DIR)
print('Original GT:', GT_DIR)
print('New analysis Output:', OUTPUT_DIR)


In [ ]:
if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch',
                    'https://github.com/hathanhv/linajea.git', str(REPO)], check=True)
COMMIT = PINNED_COMMIT or subprocess.check_output(
    ['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
subprocess.run(['git', '-C', str(REPO), 'checkout', COMMIT], check=True)
COMMIT = subprocess.check_output(
    ['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
if not (REPO / 'run_scripts' / 'analyze_biohub_pseudo_gt.py').is_file():
    raise FileNotFoundError('Push the new pseudo-GT analysis code to biohub-io first')
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'zarr>=3.1,<4', 'scipy>=1.11'], check=True)
(OUTPUT_DIR / 'commit.txt').write_text(COMMIT + '\n')
print('Code commit:', COMMIT)


## Re-solve cached candidate graphs with fixed costs

The command reads `candidate_nodes.csv`, `candidate_edges.csv`, `visible.geff`, and `split.json` from the mounted Input. For each run, it solves the same basic Linajea binary-forest ILP with costs `(7, 7)`. The first frame is taken from that run's saved ROI. The four anchor-filter policies use score thresholds 0.20/0.30/0.40/0.50, edge movement residual limits 12/8/4/3 µm, and maximum missing spans 5/4/3/2 frames. These are a **diagnostic sweep**, not tuned pseudo-GT labels.

The `anchor_strict` folder contains one provisional CSV graph, `pseudo_nodes_only.csv`, and `anchor_matches.csv` mapping selected anchor IDs to visible GT IDs. It must be audited before being added to Biohub training. The `audit_queue.csv` contains randomly sampled predictions for manual review and deliberately contains no hidden-GT verdict.


In [ ]:
command = [sys.executable, '-m', 'run_scripts.analyze_biohub_pseudo_gt',
           '--source-dir', str(OUTPUT_INPUT_DIR),
           '--output-dir', str(OUTPUT_DIR),
           '--selection-constant', str(SELECTION_CONSTANT),
           '--track-cost', str(TRACK_COST)]
if GT_DIR is not None:
    command += ['--gt-dir', str(GT_DIR)]
with (OUTPUT_DIR / 'run.log').open('w', encoding='utf-8') as log:
    process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        log.write(line); log.flush()
        if ('candidates:' in line or 'fixed ILP selected:' in line or
            'Solved components:' in line or 'summary rows' in line or
            'Traceback' in line or 'Error' in line or 'WARNING' in line):
            print(line.rstrip())
    if process.wait() != 0:
        raise RuntimeError(f'Analysis failed; inspect {OUTPUT_DIR / "run.log"}')
print('Results:', OUTPUT_DIR / 'summary.csv')


In [ ]:
summary = pd.read_csv(OUTPUT_DIR / 'summary.csv')
frames = pd.read_csv(OUTPUT_DIR / 'candidate_frames.csv')
summary['hidden_cell_recall'] = summary['hidden_recovered'] / summary['hidden_total']
summary['hidden_edge_recall'] = summary['hidden_edge_recovered'] / summary['hidden_edge_total']

cols = ['run', 'stage', 'node_count', 'edge_count', 'pseudo_node_count',
        'visible_recovered', 'visible_total', 'hidden_recovered', 'hidden_total',
        'hidden_edge_recovered', 'hidden_edge_total',
        'new_not_matched_visible', 'unconfirmed_after_full_gt',
        'hidden_confirmation_lower_bound']
display(summary[cols].sort_values(['run', 'stage']))
print('Full table:', OUTPUT_DIR / 'summary.csv')
print('Unconfirmed means unmatched by known sparse GT; it is not a false-positive count.')


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for label, group in frames.groupby('run'):
    ax.plot(group['t'], group['candidate_count'], marker='o', label=label)
ax.axhline(200, color='red', linestyle='--', label='200/frame extraction cap')
ax.set(xlabel='Frame', ylabel='Candidate nodes',
       title='Candidate generation stability by frame')
ax.legend(); ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'candidate_counts_by_frame.png', dpi=160)
display(fig)
plt.close(fig)
display(frames.groupby('run').agg(
    total_candidates=('candidate_count', 'sum'),
    capped_frames=('at_candidate_cap', 'sum'),
    median_frame_count=('candidate_count', 'median')))


In [ ]:
policy_rows = summary[summary['stage'].str.startswith('anchor_')].copy()
if GT_DIR is not None:
    fig, ax = plt.subplots(figsize=(8, 5))
    for label, group in policy_rows.groupby('run'):
        group = group.sort_values('unconfirmed_after_full_gt')
        ax.plot(group['unconfirmed_after_full_gt'], group['hidden_cell_recall'],
                marker='o', label=label)
        for _, row in group.iterrows():
            ax.annotate(row['stage'].replace('anchor_', ''),
                        (row['unconfirmed_after_full_gt'], row['hidden_cell_recall']),
                        fontsize=8)
    ax.set(xlabel='Predictions unmatched by all known GT (unknown)',
           ylabel='Hidden-cell recall at 4 µm',
           title='Recovery versus pseudo-label review burden')
else:
    fig, ax = plt.subplots(figsize=(8, 5))
    for label, group in policy_rows.groupby('run'):
        ax.scatter(group['new_not_matched_visible'], group['visible_recovered'],
                   label=label)
    ax.set(xlabel='New predictions not matched to visible GT',
           ylabel='Visible GT matched at 4 µm',
           title='Visible-support diagnostic (original GEFF not attached)')
ax.legend(); ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'recovery_vs_unconfirmed.png', dpi=160)
display(fig)
plt.close(fig)
print('Do not choose thresholds from hidden-GT recall in this exploratory plot.')


## Manual audit and next decision

`audit_queue.csv` samples predictions that passed `anchor_strict` and other new predictions. Review them on the raw image without reading the hidden-GT result. Record `true cell`, `false detection`, or `uncertain` in `manual_cell_review`; review temporal links separately before using any pseudo-edge. An unmatched prediction is not automatically a false positive. The policy graphs are only **provisional pseudo-GT candidates**.


In [ ]:
audit = pd.read_csv(OUTPUT_DIR / 'audit_queue.csv')
display(audit.groupby(['run', 'group']).size().rename('review_items').to_frame())
display(audit.head(10))
image_path = GT_DIR / f'{SAMPLE}.zarr' if GT_DIR is not None else None
if image_path is not None and image_path.is_dir() and not audit.empty:
    import zarr
    image = zarr.open_group(str(image_path), mode='r')['0']
    preview = audit[audit['group'] == 'anchored_pseudo'].head(9)
    if not preview.empty:
        fig, axes = plt.subplots(3, 3, figsize=(9, 9))
        for ax, (_, row) in zip(axes.flat, preview.iterrows()):
            t, z, y, x = (int(row[k]) for k in ('t', 'z', 'y', 'x'))
            z0, z1 = max(0, z - 2), min(image.shape[1], z + 3)
            y0, y1 = max(0, y - 24), min(image.shape[2], y + 25)
            x0, x1 = max(0, x - 24), min(image.shape[3], x + 25)
            patch = np.asarray(image[t, z0:z1, y0:y1, x0:x1]).max(axis=0)
            ax.imshow(patch, cmap='gray')
            ax.scatter([x - x0], [y - y0], s=65, facecolors='none', edgecolors='lime')
            ax.set_title(f"{row['run']} t={t} id={int(row['node_id'])}", fontsize=8)
            ax.axis('off')
        for ax in axes.flat[len(preview):]:
            ax.axis('off')
        fig.tight_layout()
        fig.savefig(OUTPUT_DIR / 'audit_preview.png', dpi=160)
        display(fig)
        plt.close(fig)
print('Manual audit queue:', OUTPUT_DIR / 'audit_queue.csv')
print('All outputs:', OUTPUT_DIR)
